# Práctica 1: Aprendizaje Automático
**Predicción de Subscripsción a un Producto Bancario**

* **Autores:** Jaime del Campo Montaña & David Alexander Torres Paillacho
* **NIA's:** 100495957 100522187

In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn
#necesario para intalar la librería numpy, que es una dependencia de scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------------------------- -------------- 6.3/9.7 MB 45.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 38.4 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Jaime\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn
#necesario para instalr la libreía seaborn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Jaime\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

RANDOM_STATE = 100522187
np.random.seed(RANDOM_STATE)


In [7]:
dataframe = pd.read_pickle('bank_15.pkl')
#muestro las primeras filas del dataframe --> comprobación
display(dataframe.head())

#flas, columnas, tipos de datos y valores nulos
print("\nInformación del dataframe")
dataframe.info()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes



Información del dataframe
<class 'pandas.DataFrame'>
Index: 11000 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11000 non-null  int64 
 1   job        10920 non-null  object
 2   marital    11000 non-null  object
 3   education  10744 non-null  object
 4   default    11000 non-null  object
 5   balance    11000 non-null  int64 
 6   housing    11000 non-null  object
 7   loan       11000 non-null  object
 8   contact    11000 non-null  object
 9   day        11000 non-null  int64 
 10  month      11000 non-null  object
 11  duration   11000 non-null  int64 
 12  campaign   11000 non-null  int64 
 13  pdays      11000 non-null  int64 
 14  previous   11000 non-null  int64 
 15  poutcome   11000 non-null  object
 16  deposit    11000 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.5+ MB


A continuación analizaremos los datos del dataset


In [8]:
#número de instancias y variables
print(f"número de instancias (filas): {dataframe.shape[0]}")
print(f"número de variables (columnas): {dataframe.shape[1]}")
#comprobación de valores nulos
print("\nvalores null por columna")
nulos = dataframe.isnull().sum()
print(nulos[nulos > 0])

número de instancias (filas): 11000
número de variables (columnas): 17

valores null por columna
job           80
education    256
dtype: int64


In [9]:
#variables categóricas > 10
categorical_columans = dataframe.select_dtypes(include=['object']).columns
print("\nCardinalidad de variables categóricas")
for columna in categorical_columans:
    val_unicos = dataframe[columna].nunique()
    if val_unicos > 10:
        print(f"'{columna}'tiene alta cardinalidad: {val_unicos} valores únicos")
    else:
        print(f"'{columna}':{val_unicos} valores únicos")

#columnas constantes
print("\ncolumnas constantes")
constant_columnas = [columna for columna in dataframe.columns if dataframe[columna].nunique() <= 1]
if len(constant_columnas) == 0:
    print("no hay columnas constantes.")
else:
    print(f"columnas constantes encontradas: {constant_columnas}")


Cardinalidad de variables categóricas
'job'tiene alta cardinalidad: 12 valores únicos
'marital':3 valores únicos
'education':4 valores únicos
'default':2 valores únicos
'housing':2 valores únicos
'loan':2 valores únicos
'contact':3 valores únicos
'month'tiene alta cardinalidad: 12 valores únicos
'poutcome':4 valores únicos
'deposit':2 valores únicos

columnas constantes
no hay columnas constantes.


In [10]:
# Comprobar desbalanceo de la variable objetivo
print("Distribución de la variable objetivo (deposit):")
print(dataframe['deposit'].value_counts(normalize=True))

# Análisis de pdays
print("\nEstadísticas de pdays:")
print(dataframe['pdays'].describe())
# Tip: Muchos modelos sufren con el -1 si el resto son valores altos. 
# Podrías crear una columna binaria 'no_previo' si pdays == -1.

Distribución de la variable objetivo (deposit):
deposit
no     0.525455
yes    0.474545
Name: proportion, dtype: float64

Estadísticas de pdays:
count    11000.000000
mean        51.308636
std        108.782842
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         20.250000
max        854.000000
Name: pdays, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split

# Separar X e y
X = dataframe.drop('deposit', axis=1)
y = dataframe['deposit']

# División Holdout (2/3 train, 1/3 test) según instrucciones
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=RANDOM_STATE, stratify=y
)

print(f"Tamaño entrenamiento: {X_train.shape[0]}")
print(f"Tamaño test: {X_test.shape[0]}")

Tamaño entrenamiento: 7370
Tamaño test: 3630
